In [ ]:
import os
import torch
import timm
import qoi  # QOI 디코더 모듈 (qoi.py 필요)
import numpy as np
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, TensorDataset


# ------------------------------
# 🔹 QOI 파일 로더 (with 캐시)
# ------------------------------
qoi_cache = {}

def qoi_loader(qoi_path):
    if qoi_path in qoi_cache:
        return qoi_cache[qoi_path]

    if not os.path.exists(qoi_path):
        print(f"🚨 파일 없음: {qoi_path}")
        return None

    with open(qoi_path, "rb") as f:
        qoi_data = f.read()

    img = qoi.decode(qoi_data)
    img = Image.fromarray(img).convert("RGB")

    qoi_cache[qoi_path] = img
    return img


# ------------------------------
# 🔹 EfficientNet-B4 모델 정의
# ------------------------------
class EfficientNetB4Fine(torch.nn.Module):
    def __init__(self, num_classes, pretrained_weights_path):
        super(EfficientNetB4Fine, self).__init__()
        self.EfficientNetB4Fine = timm.create_model("efficientnet_b4", pretrained=False, num_classes=num_classes)
        state_dict = torch.load(pretrained_weights_path, map_location="cpu")
        self.EfficientNetB4Fine.load_state_dict(state_dict)

    def forward(self, x):
        return self.EfficientNetB4Fine(x)


# ------------------------------
# 🔹 테스트 평가 및 틀린 샘플 수집
# ------------------------------
def evaluate_and_collect_wrongs(model, data_loader, device):
    model.eval()
    correct, total = 0, 0
    wrong_images = []
    wrong_labels = []

    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc="Testing", unit="batch"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            for img, pred, label in zip(images, preds, labels):
                if pred != label:
                    wrong_images.append(img.cpu())
                    wrong_labels.append(label.cpu())

    acc = 100 * correct / total
    print(f"✅ EfficientNet-B4 테스트 정확도: {acc:.2f}%")
    return acc, wrong_images, wrong_labels


# ------------------------------
# 🔹 틀린 샘플로 재학습
# ------------------------------
def fine_tune_on_wrongs(model, wrong_images, wrong_labels, device, epochs=3, lr=1e-4, save_path="fine_tuned_model.pth"):
    if not wrong_images:
        print("🎉 틀린 샘플이 없어 재학습할 필요가 없습니다.")
        return

    print(f"🔁 재학습 시작: {len(wrong_images)}개 샘플")

    dataset = TensorDataset(torch.stack(wrong_images), torch.tensor(wrong_labels))
    loader = DataLoader(dataset, batch_size=16, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        pbar = tqdm(loader, desc=f"Fine-tuning Epoch {epoch+1}/{epochs}", unit="batch")
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            pbar.set_postfix(loss=running_loss / (pbar.n + 1))

    print("✅ 재학습 완료")

    # 모델 저장
    
    print(device)
    torch.save(model.state_dict(), save_path)
    print(f"💾 재학습된 모델 저장 완료: {save_path}")



# ------------------------------
# 🔹 메인 함수
# ------------------------------
def main():
    # 설정값
    num_classes = 64
    batch_size = 32
    num_workers = 0
    pretrained_path = "/root/Public_Storage/madelab_khw/lpcv/model/better_efficient_b4 (1).pth"
    test_path = "/root/Public_Storage/madelab_khw/lpcv/coco/cropped_dir_test"
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # 테스트 전처리 정의
    transform_test = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # QOI 기반 테스트셋 생성
    test_dataset = ImageFolder(
        root=test_path,
        transform=transform_test,
        loader=qoi_loader,
        is_valid_file=lambda path: path.endswith(".qoi")
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    # 모델 로드 및 초기 평가
    model = EfficientNetB4Fine(num_classes=num_classes, pretrained_weights_path=pretrained_path)
    model.to(device)

    test_acc, wrong_images, wrong_labels = evaluate_and_collect_wrongs(model, test_loader, device)
    
    # 틀린 샘플로 재학습
    fine_tune_on_wrongs(
        model, wrong_images, wrong_labels, device,
        epochs=3, lr=1e-4,
        save_path="fine_tuned_model.pth"  # 저장 파일명
    )

    # 재평가
    print("\n🔍 재학습 후 정확도 재평가")
    evaluate_and_collect_wrongs(model, test_loader, device)


# ------------------------------
# 🔹 실행
# ------------------------------
if __name__ == "__main__":
    main()


/tmp/ipykernel_3592063/3041619015.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_weights_path, map_location="cpu")


odict_keys(['conv_stem.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'blocks.0.0.conv_dw.weight', 'blocks.0.0.bn1.weight', 'blocks.0.0.bn1.bias', 'blocks.0.0.bn1.running_mean', 'blocks.0.0.bn1.running_var', 'blocks.0.0.bn1.num_batches_tracked', 'blocks.0.0.se.conv_reduce.weight', 'blocks.0.0.se.conv_reduce.bias', 'blocks.0.0.se.conv_expand.weight', 'blocks.0.0.se.conv_expand.bias', 'blocks.0.0.conv_pw.weight', 'blocks.0.0.bn2.weight', 'blocks.0.0.bn2.bias', 'blocks.0.0.bn2.running_mean', 'blocks.0.0.bn2.running_var', 'blocks.0.0.bn2.num_batches_tracked', 'blocks.0.1.conv_dw.weight', 'blocks.0.1.bn1.weight', 'blocks.0.1.bn1.bias', 'blocks.0.1.bn1.running_mean', 'blocks.0.1.bn1.running_var', 'blocks.0.1.bn1.num_batches_tracked', 'blocks.0.1.se.conv_reduce.weight', 'blocks.0.1.se.conv_reduce.bias', 'blocks.0.1.se.conv_expand.weight', 'blocks.0.1.se.conv_expand.bias', 'blocks.0.1.conv_pw.weight', 'blocks.0.1.bn2.weight', 'blocks.0.1.bn

Testing:  12% 87/721 [00:09<01:17,  8.13batch/s]